**KEYPOINTS DETECTOR**

In [24]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools

**DESCRIPTORS**

In [ ]:
def sift_descriptor(
    img: np.ndarray,
    params
):
    if params is None:
        params = {}

    sift = cv2.SIFT_create(
        nfeatures=params.get("nfeatures", 0),
        nOctaveLayers=params.get("nOctaveLayers", 3),
        contrastThreshold=params.get("contrastThreshold", 0.04),
        edgeThreshold=params.get("edgeThreshold", 10),
        sigma=params.get("sigma", 1.6),
    )

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    keypoints, descriptors = sift.detectAndCompute(gray, None)
    return keypoints, descriptors


def orb_descriptor(
    img: np.ndarray,
    params
):
    if params is None:
        params = {}

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img

    orb = cv2.ORB_create(
        nfeatures=params.get("nfeatures", 500),
        scaleFactor=params.get("scaleFactor", 1.2),
        nlevels=params.get("nlevels", 8),
        edgeThreshold=params.get("edgeThreshold", 31),
        firstLevel=params.get("firstLevel", 0),
        WTA_K=params.get("WTA_K", 2),
        scoreType=params.get("scoreType", cv2.ORB_HARRIS_SCORE),
        patchSize=params.get("patchSize", 31),
        fastThreshold=params.get("fastThreshold", 20),
    )

    keypoints, descriptors = orb.detectAndCompute(gray, None)
    return keypoints, descriptors


def color_sift_descriptor(
    img: np.ndarray,
    params
):
    if params is None:
        params = {}

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    sift = cv2.SIFT_create(
        nfeatures=params.get("nfeatures", 0),
        nOctaveLayers=params.get("nOctaveLayers", 3),
        contrastThreshold=params.get("contrastThreshold", 0.04),
        edgeThreshold=params.get("edgeThreshold", 10),
        sigma=params.get("sigma", 1.6),
    )

    keypoints = sift.detect(v, None)

    desc_h = np.zeros((len(keypoints), 128))
    desc_s = np.zeros_like(desc_h)
    desc_v = np.zeros_like(desc_h)

    if keypoints:
        _, desc_h = sift.compute(h, keypoints)
        _, desc_s = sift.compute(s, keypoints)
        _, desc_v = sift.compute(v, keypoints)

    descriptors = np.concatenate((desc_h, desc_s, desc_v), axis=1)
    return keypoints, descriptors


**PRECOMPUTATION DB DESCRIPTORS USING DIFFERENT HYPERPARAMETERS**

In [ ]:
bbdd_path = "../data/BBDD"
current_dir = os.getcwd()
output_root = os.path.join(current_dir, "BBDD_DESCRIPTORS")
os.makedirs(output_root, exist_ok=True)

descriptor_funcs = {
    "sift": sift_descriptor,
    "orb": orb_descriptor,
    "color_sift": color_sift_descriptor
}

param_grids = {
    "sift": {
        "sigma": [1.2, 1.6],
        "edgeThreshold": [6,8, 10],
        "nOctaveLayers": [3, 5],
    },
    "orb": {
        "nfeatures": [500, 1000],
        "fastThreshold": [10, 20],
    },
    "color_sift": {
        "sigma": [1.4, 1.6],
        "edgeThreshold": [6,8, 10],
        "nOctaveLayers": [3, 5],
    },
}

image_files = [f for f in os.listdir(bbdd_path) if f.lower().endswith((".jpg"))]

for descriptor, func in descriptor_funcs.items():
    print(f"Processing: {descriptor.upper()}")

    param_grid = param_grids.get(descriptor, {})

    if not param_grid:
        param_combinations = [{}]
    else:
        keys, values = zip(*param_grid.items())
        param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    for params in param_combinations:

        param_name = "_".join([f"{k}{v}" for k, v in params.items()]) if params else "default"
        folder_name = f"BBDD_{descriptor}_{param_name}"
        output_dir = os.path.join(output_root, folder_name)
        os.makedirs(output_dir, exist_ok=True)

        print(f"\n{descriptor.upper()} with {params}")

        for img_name in image_files:
            img_path = os.path.join(bbdd_path, img_name)
            img = cv2.imread(img_path)

            keypoints, descriptors = func(img, params=params)
            coords = np.array([kp.pt for kp in keypoints], dtype=np.float32)

            base_name = os.path.splitext(img_name)[0]
            save_path = os.path.join(output_dir, f"{base_name}_{descriptor}.npz")
            np.savez_compressed(save_path, keypoints=coords, descriptors=descriptors)

        print(f"Saved in {output_dir}")



🔹 Procesando descriptor: SIFT

SIFT con {'contrastThreshold': 0.02, 'sigma': 1.2, 'edgeThreshold': 8}
Guardado en c:\Users\xavipba\OneDrive\Escritorio\Msc Computer vision\C1 Project\Team3\exploratory_analysis\BBDD_DESCRIPTORS\BBDD_sift_contrastThreshold0.02_sigma1.2_edgeThreshold8

SIFT con {'contrastThreshold': 0.02, 'sigma': 1.2, 'edgeThreshold': 10}


KeyboardInterrupt: 

FOR EACH DESCRIPTOR COMBINATION, GENERATE THE DEVELOPMENT SET DESCRIPTOR, SELECT THE PRECOMPUTED DB DESCRIPTORS, HYPERPARAMETER SEARCH USING DIFFERENT SIMILARITY METRICS / FILTERING OF KEYPOINTS / FINAL SCORE